# VMGP: Variational Auto-Encoder Multi-task Genomic Prediction
## Training & Risultati

Questo notebook usa il modulo `vae/` per training e analisi.

La logica riutilizzabile è stata estratta in moduli Python:

- `vae/experiment.py` — `run_grid`, `run_single` (orchestrazione)
- `vae/plotting.py` — funzioni di visualizzazione
- `vae/attribution.py` — helper SHAP / Random Forest
- `vae/results_io.py` — salvataggio incrementale risultati

Il notebook è ora un livello di orchestrazione/presentazione.

### Struttura del Modulo `vae/`:
- `vae/network.py` - Architettura VMGP
- `vae/lightning_module.py` - Logic training (PyTorch Lightning)
- `vae/data_module.py` - Caricamento e preprocessing dati PLINK
- `vae/utils.py` - Funzioni di utilità (balancing, capping, metriche)
- `vae/experiment.py` - Runner esperimenti con K-Fold CV
- `vae/analysis.py` - Analisi post-training (legacy, non usato qui)

### Esperimento:
Griglia completa di **20 combinazioni**:
- **Min Samples per Razza:** 30, 40, 50, 60, 70 (solo sample reali, NO SMOTE)
- **Dimensione Latent Space:** 32, 64, 96, 128
- **Task:** Solo classificazione breed

---


## 1. Setup e Imports

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import pytorch_lightning as pl

# Aggiungi la directory parent al path per importare il modulo vae e genomic
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Imposta seed per riproducibilità
pl.seed_everything(42)

from vae import VMGP_LightningSystem
from vae.experiment import run_grid, run_single
from vae.plotting import (
    plot_grid_heatmaps,
    plot_tradeoff,
    plot_trends,
    plot_cm_grid,
    plot_single_cm,
)
from vae.attribution import (
    BreedWrapper,
    aggregate_shap_to_global,
    run_shap_accuracy_curve,
    run_rf_selection,
)

from genomic import load_split, GenomicExperimentData

print("✅ Moduli caricati correttamente!")
print(f"🔧 CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")



## 2. Configurazione

In [ ]:
# ⚠️ MODIFICA QUESTI PATH CON IL TUO DATASET ⚠️
BED_PATH = "../data/ADAPTmap_genotypeTOP_20160222_full.bed"
FAM_PATH = "../data/ADAPTmap_genotypeTOP_20160222_full.fam"
METADATA_PATH = "classificazione-capre.csv"

# Hyperparameters
config = {
    # Data
    'metadata_path': METADATA_PATH,
    'batch_size': 64,

    # Balancing (solo sul TRAIN trasformato; politica invariata)
    'target_samples': 30,

    # Model (VMGP) - Loss weights
    'alpha': 0.5,
    'lr': 0.0001,
    'latent_dim': 32,  # Default, verrà variato negli esperimenti

    # Solo classificatore breed
    'weight_breed': 1.0,
    'weight_continent': 0.0,
    'weight_caseina': 0.0,

    # Training
    'max_epochs': 100,
    'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
    'devices': 1,
}

# Protocollo v2: coorte primaria + split congelati (nessun QC/split locale).
# Il preprocessing (marker QC/MAF/imputazione/LD) e i fold provengono dal layer
# condiviso genomic.experiment_data, fit solo sui dati di training.
split = load_split(
    "ADAPTmap_genotypeTOP_20160222_full",
    outer_split_seed=42,
    fold_seed=42,
    fam_path=FAM_PATH,
    out_dir="../splits",
    cohort_label="indmiss0p10",
)
experiment_data = GenomicExperimentData.from_plink(split, BED_PATH)

# Griglia esperimenti (solo latent_dim: la coorte è congelata dal protocollo v2)
LATENT_DIM_RANGE = [32, 64, 96, 128]        # 4 valori

print("📋 CONFIGURAZIONE BASE:")
for k, v in config.items():
    print(f"   {k}: {v}")

print(f"\n📊 GRIGLIA ESPERIMENTI:")
print(f"   Latent Dims: {LATENT_DIM_RANGE}")
print(f"   Coorte: {experiment_data.n_samples} animali, {experiment_data.n_classes} razze")
print(f"   Task: Solo classificazione breed (NO continente, NO caseina)")
print(f"   Dati: Solo sample reali (NO SMOTE)")



## 3. Grid Experiment

In [ ]:
# === ESPERIMENTO: GRIGLIA LATENT DIM ===
# Coorte congelata (protocollo v2); si varia solo latent_dim.
# Solo breed classification, solo dati reali (cap, NO SMOTE).

grid_results, all_results = run_grid(experiment_data, config, LATENT_DIM_RANGE)



### 3.1 Grid results / figures

In [ ]:
# === ANALISI GRIGLIA LATENT DIM ===
df_grid = pd.DataFrame(grid_results)

# Metriche da visualizzare (include Kappa per confronto equo)
metrics_to_plot = {
    'Acc_Breed (mean)': 'Breed Accuracy (raw)',
    'Kappa_Breed (mean)': "Cohen's Kappa (chance-adjusted)",
    'Silhouette (mean)': 'Silhouette Score (Clustering)',
    'Generalization (GE) (mean)': 'Generalization (GE)'
}
metrics_to_plot = {k: v for k, v in metrics_to_plot.items() if k in df_grid.columns}

import matplotlib.pyplot as plt

n_metrics = len(metrics_to_plot)
fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5))
if n_metrics == 1:
    axes = [axes]
for ax, (metric_col, metric_name) in zip(axes, metrics_to_plot.items()):
    ordered = df_grid.sort_values('Latent_Dim')
    ax.bar(ordered['Latent_Dim'].astype(str), ordered[metric_col], color='steelblue')
    ax.set_title(metric_name, fontweight='bold', fontsize=12)
    ax.set_xlabel('Latent Dimension', fontsize=11)
    ax.set_ylabel(metric_name, fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('📊 Griglia Latent Dim (coorte congelata: 34 razze, 3 fold CV)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabella riassuntiva
print("\n📋 TABELLA COMPLETA DEI RISULTATI:")
display_cols = ['Experiment', 'Latent_Dim', 'Num Classes Breed', 'Num SNPs'] + list(metrics_to_plot.keys())
available_cols = [c for c in display_cols if c in df_grid.columns]
print(df_grid[available_cols].to_string())

# Identificazione miglior latent_dim
if 'Kappa_Breed (mean)' in df_grid.columns:
    best_idx = df_grid['Kappa_Breed (mean)'].idxmax()
    best = df_grid.loc[best_idx]
    print(f"\n🎯 MIGLIOR LATENT DIM (per Cohen's Kappa): LatDim={int(best['Latent_Dim'])} "
          f"| Kappa={best['Kappa_Breed (mean)']:.4f} | Acc={best.get('Acc_Breed (mean)', float('nan')):.4f}")

if 'Acc_Breed (mean)' in df_grid.columns:
    best_acc_idx = df_grid['Acc_Breed (mean)'].idxmax()
    best_acc = df_grid.loc[best_acc_idx]
    print(f"🎯 MIGLIOR LATENT DIM (per Accuracy raw): LatDim={int(best_acc['Latent_Dim'])} "
          f"| Acc={best_acc['Acc_Breed (mean)']:.4f}")



### 3.2 Statistiche per latent dim


In [ ]:
# === STATISTICHE PER LATENT DIM (media ± std sui 3 fold CV) ===
metric_keys = ['Acc_Breed (mean)', 'Kappa_Breed (mean)',
               'Generalization (GE) (mean)', 'Silhouette (mean)']
metric_keys = [m for m in metric_keys if m in df_grid.columns]

summary_rows = []
for _, row in df_grid.sort_values('Latent_Dim').iterrows():
    r = {'Latent_Dim': int(row['Latent_Dim'])}
    for m in metric_keys:
        std_key = m.replace(' (mean)', ' (std)')
        std_val = row.get(std_key, float('nan'))
        r[m.replace(' (mean)', '')] = f"{row[m]:.4f} ± {std_val:.4f}"
    summary_rows.append(r)
summary_df = pd.DataFrame(summary_rows)
print("\n📊 STATISTICHE PER LATENT DIM (media ± std sui fold):")
print(summary_df.to_string(index=False))



### 3.3 Trend per latent dim

In [ ]:
# === TREND DELLE METRICHE AL VARIARE DI LATENT DIM ===
import matplotlib.pyplot as plt

metrics_lines = {
    'Kappa_Breed (mean)': "Cohen's Kappa (chance-adjusted)",
    'Acc_Breed (mean)': 'Breed Accuracy (raw)',
    'Generalization (GE) (mean)': 'Generalization (GE)',
    'Silhouette (mean)': 'Silhouette Score',
}
metrics_lines = {k: v for k, v in metrics_lines.items() if k in df_grid.columns}

ordered = df_grid.sort_values('Latent_Dim')
n_metrics = len(metrics_lines)
fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5))
if n_metrics == 1:
    axes = [axes]
colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D"]
for ax, (metric_col, metric_name), color in zip(axes, metrics_lines.items(), colors):
    ax.plot(ordered['Latent_Dim'], ordered[metric_col], 'o-', color=color,
            linewidth=2, markersize=8)
    ax.set_xlabel('Latent Dimension', fontsize=11)
    ax.set_ylabel(metric_name, fontsize=11)
    ax.set_title(metric_name, fontweight='bold')
    ax.set_xticks(ordered['Latent_Dim'].tolist())
    ax.grid(True, alpha=0.3)
plt.suptitle('📈 Trend per Dimensione Latente', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



### 3.4 Conclusioni

In [ ]:
# === CONCLUSIONI GRIGLIA ESPERIMENTI ===
print("🎯 ANALISI DEI RISULTATI (grid latent dim con 3-Fold CV)")
print("="*60)

if 'Kappa_Breed (mean)' in df_grid.columns:
    top_kappa = df_grid.sort_values('Kappa_Breed (mean)', ascending=False)
    print("\n🏆 CLASSIFICA CONFIGURAZIONI (per Cohen's Kappa):")
    for i, (_, row) in enumerate(top_kappa.iterrows(), 1):
        print(f"   {i}. LatDim={int(row['Latent_Dim'])} "
              f"→ Kappa={row['Kappa_Breed (mean)']:.4f}, Acc={row.get('Acc_Breed (mean)', 0):.4f}")

if 'Acc_Breed (mean)' in df_grid.columns:
    top_acc = df_grid.sort_values('Acc_Breed (mean)', ascending=False)
    print("\n🏆 CLASSIFICA CONFIGURAZIONI (per Accuracy raw):")
    for i, (_, row) in enumerate(top_acc.iterrows(), 1):
        print(f"   {i}. LatDim={int(row['Latent_Dim'])} → Acc={row['Acc_Breed (mean)']:.4f}")

print("\n" + "="*60)
print("📝 NOTE:")
print("   • Coorte primaria congelata (protocollo v2): 2863 animali, 34 razze, 3 fold CV")
print("   • Solo classificazione breed; solo sample reali (NO dati sintetici)")
print("   • Il preprocessing genomico è fit sui soli fold di training e riusato tra latent dim")
print("   • Cohen's Kappa è la metrica preferita (chance-adjusted)")
print("   • Le confusion matrix sono in results/confusion_matrices/")

os.makedirs('results', exist_ok=True)
export_cols = [c for c in df_grid.columns if c not in ['Confusion Matrix Breed', 'Class Names Breed']]
df_grid[export_cols].to_csv('results/grid_experiment_results.csv', index=False)
print("\n💾 Risultati salvati in results/grid_experiment_results.csv")



### 3.5 Confusion matrices

In [ ]:
# === CONFUSION MATRIX PER OGNI LATENT DIM ===
import matplotlib.pyplot as plt
import seaborn as sns

for res in grid_results:
    cm = res.get('Confusion Matrix Breed')
    if cm is None:
        continue
    class_names = res.get('Class Names Breed', [str(i) for i in range(len(cm))])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    latent_dim = res.get('Latent_Dim', '?')
    acc = res.get('Acc_Breed (mean)', 0)
    kappa = res.get('Kappa_Breed (mean)', 0)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=False, cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                vmin=0, vmax=1, linewidths=0.3, linecolor='gray',
                cbar_kws={'label': 'Recall per classe'})
    ax.set_title(f'Confusion Matrix — LatDim={latent_dim}\nAcc={acc:.3f} | κ={kappa:.3f}',
                 fontweight='bold', fontsize=13)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='both', labelsize=7)
    plt.tight_layout()
    os.makedirs('results/confusion_matrices', exist_ok=True)
    plt.savefig(f'results/confusion_matrices/cm_grid_latdim_{latent_dim}.png', dpi=150, bbox_inches='tight')
    plt.show()

print("✅ Confusion matrix visualizzate per ogni latent dim")



## 4. Single Experiment

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURA QUI
# ═══════════════════════════════════════════════════
SINGLE_LATENT_DIM  = 96    # Dimensione del latent space
# ═══════════════════════════════════════════════════

print(f"Configurazione singolo esperimento:")
print(f"   Latent Dim:  {SINGLE_LATENT_DIM}")



In [ ]:
# === TRAINING SINGOLO ESPERIMENTO ===
single_result, exp_name = run_single(experiment_data, config, SINGLE_LATENT_DIM)



### 4.1 Single-experiment results

In [ ]:
# === RISULTATI SINGOLO ESPERIMENTO ===
print(f"📊 RISULTATI: {exp_name}")
print(f"{'='*60}")
print(f"   Razze:       {single_result.get('Num Classes Breed', '?')}")
print(f"   SNPs:        {single_result.get('Num SNPs', '?')}")
print(f"   Best Fold:   {single_result.get('Best Fold', '?')}")

# Stampa tutte le metriche (mean ± std)
print(f"\n   Metriche (mean ± std su {experiment_data.split.n_folds} fold):")
for key in sorted(single_result.keys()):
    if key.endswith('(mean)'):
        std_key = key.replace('(mean)', '(std)')
        mean_val = single_result[key]
        std_val = single_result.get(std_key, 0)
        metric_name = key.replace(' (mean)', '')
        print(f"   {metric_name:30s}: {mean_val:.4f} ± {std_val:.4f}")

# Confusion Matrix
plot_single_cm(single_result, exp_name)


## 5. SHAP analysis

> ⚠️ **Nota metodologica (intenzionalmente NON corretta qui).**
> L'attribuzione SHAP/RF è calcolata sull'**intero dataset** (non solo
> train): questo è un noto problema di leakage che verrà corretto in una
> fase successiva.

Studio analogo al modello Contrastivo: verificare se i Top-% SNP identificati
dagli SHAP del classificatore breed del VMGP sono sufficienti per classificare
la razza meglio di una selezione casuale.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEZIONE 5 — SHAP SNP Pre-selection (protocollo paper VMGP)
# ═══════════════════════════════════════════════════════════════
import shap
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import EarlyStopping

# ── CONFIG (riusa il DataModule già caricato per min_samples=30) ──
SHAP_MIN_SAMPLES = 30
SHAP_LATENT_DIM  = 96       # usa la tua configurazione migliore
SHAP_EPOCHS      = 100
SEED             = 42
os.makedirs('results/shap_vmgp', exist_ok=True)

# ── 1. Carica DataModule dedicato per SHAP ────────────────────────
dm = GenomicDataModule(
    bed_path=BED_PATH,
    metadata_path=METADATA_PATH,
    batch_size=64,
    maf_thresh=0.01,
    geno_thresh=0.1,
    min_samples_per_class=SHAP_MIN_SAMPLES,
    ld_pruning=True,
    ld_threshold=0.2,
    ld_window=50,
    val_split=0.15,
    test_split=0.15,
    use_breed=True,
    use_continent=False,
    use_caseina=False,
    use_attitudine=False
)
dm.setup()

X_all = dm.X_processed
y_all = dm.y_breed_processed
print(f"Dataset completo: {X_all.shape[0]} campioni | {X_all.shape[1]} SNP | {dm.num_classes_breed} razze")

# ── 2. Split 10% ps / 90% validation (stratificato per razza) ────
ps_idx, val90_idx = train_test_split(
    np.arange(len(X_all)),
    test_size=0.90, stratify=y_all, random_state=SEED
)
X_ps,  y_ps  = X_all[ps_idx],     y_all[ps_idx]
X_90,  y_90  = X_all[val90_idx],  y_all[val90_idx]
print(f"ps (10%): {len(X_ps)} campioni | val90: {len(X_90)} campioni")

# ── 4. Training modello ps ────────────────────────────────────────
print("\n🔧 Training modello ps (10%)...")

# Dummy y_cont/cas/att (non usati — breed_only)
y_dummy = np.zeros(len(y_ps), dtype=np.int64)

ps_ds = TensorDataset(
    torch.FloatTensor(X_ps),
    torch.LongTensor(y_ps),
    torch.LongTensor(y_dummy),  # continent
    torch.LongTensor(y_dummy),  # caseina
    torch.LongTensor(y_dummy),  # attitudine
)
# Split interno 85/15 per l'early stopping
n_val_ps = max(int(0.15 * len(ps_ds)), 1)
n_train_ps = len(ps_ds) - n_val_ps
ps_train_ds, ps_val_ds = torch.utils.data.random_split(
    ps_ds, [n_train_ps, n_val_ps],
    generator=torch.Generator().manual_seed(SEED)
)

ps_train_loader = DataLoader(ps_train_ds, batch_size=32, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
ps_val_loader   = DataLoader(ps_val_ds,   batch_size=32,
                              num_workers=4, pin_memory=True)

model_ps = VMGP_LightningSystem(
    num_snps=dm.num_snps,
    num_classes_breed=dm.num_classes_breed,
    num_classes_continent=0,
    num_classes_caseina=0,
    num_classes_attitudine=0,
    alpha=0.5, lr=0.0001,
    use_breed=True, use_continent=False, use_caseina=False, use_attitudine=False,
    weight_breed=1.0, weight_continent=0.0, weight_caseina=0.0, weight_attitudine=0.0,
    latent_dim=SHAP_LATENT_DIM
)

early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min', verbose=False)
trainer_ps = pl.Trainer(
    precision="16-mixed", max_epochs=SHAP_EPOCHS,
    accelerator='gpu', devices=1,
    callbacks=[early_stop],
    enable_progress_bar=True, enable_model_summary=False,
    enable_checkpointing=False, logger=False
)
trainer_ps.fit(model_ps, ps_train_loader, ps_val_loader)
print("✅ Training ps completato.")

# ── 5. Calcolo SHAP values ─────────────────────────────────────────
wrapper = BreedWrapper(model_ps).eval().to('cuda')


X_ps_t = torch.FloatTensor(X_ps).to('cuda')
# Background: 100 campioni random dal ps (come da paper)
np.random.seed(SEED)
bg_idx  = np.random.choice(len(X_ps), min(100, len(X_ps)), replace=False)
X_bg_t  = X_ps_t[bg_idx]

print(f"\n🔍 Calcolo SHAP su {len(X_ps)} campioni ps...")
explainer   = shap.GradientExplainer(wrapper, X_bg_t)
shap_values = explainer.shap_values(X_ps_t)   # list o ndarray, dipende dalla versione SHAP

# ── 6. Aggregazione robusta (helper estratto) ─────────────────────
N_SNPS = X_all.shape[1]
shap_global = aggregate_shap_to_global(shap_values, N_SNPS)

np.save('results/shap_vmgp/shap_global.npy', shap_global)
print(f"✅ SHAP global salvato. Shape: {shap_global.shape}")

# Top 10 SNP
top10 = np.argsort(shap_global)[::-1][:10]
print("\nTop 10 SNP per SHAP aggregato:")
for r, idx in enumerate(top10, 1):
    print(f"  {r:2d}. SNP[{int(idx)}] = {float(shap_global[idx]):.6f}")

# ── 7. Curva accuracy per soglia di SNP (helper estratto) ─────────
df_shap = run_shap_accuracy_curve(shap_global, X_90, y_90, seed=SEED)


## 6. RF analysis

In [ ]:
# Random forest approach (helper estratto)
df_rf = run_rf_selection(X_all, y_all, X_ps, y_ps, X_90, y_90, seed=SEED)


## 7. Interpretation

Analisi e conclusioni qualitative sui risultati delle sezioni precedenti
(vanno inserite qui in base agli output di training).
